In [20]:
import requests
import pandas as pd

url = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json'

# Melakukan request ke Socrata API (biasanya menggunakan format .json untuk resource)
try:
    # Kita batasi limit awal 1000 data untuk contoh
    response = requests.get(url, params={"$limit": 1000})
    response.raise_for_status()
    data = response.json()

    # Mengubah ke DataFrame
    df_nyc = pd.DataFrame(data)
    print(f"Berhasil memuat {len(df_nyc)} baris data.")
    display(df_nyc.head())
except Exception as e:
    print(f"Terjadi kesalahan saat mengambil data: {e}")

Berhasil memuat 1000 baris data.


,unique_key,created_date,agency,agency_name,complaint_type,descriptor,incident_zip,intersection_street_1,intersection_street_2,address_type,...,landmark,bbl,vehicle_type,descriptor_2,taxi_pick_up_location,closed_date,bridge_highway_name,bridge_highway_direction,bridge_highway_segment,road_ramp
0,70709892,2026-10-11T02:42:35.000,DOT,Department of Transportation,Street Condition,Pothole,11201,FULTON STREET,GALLATIN PLACE,INTERSECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,70720084,2026-10-11T01:51:04.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,70719843,2026-10-11T01:50:47.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,10011,7 AVENUE,8 AVENUE,ADDRESS,...,WEST 21 STREET,1007710013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,70719844,2026-10-11T01:50:36.000,NYPD,New York City Police Department,Noise - Commercial,Loud Talking,10001,7 AVENUE,8 AVENUE,ADDRESS,...,WEST 28 STREET,1007780020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,70720001,2026-10-11T01:50:36.000,NYPD,New York City Police Department,Noise - Commercial,Loud Music/Party,11373,BROADWAY,41 AVENUE,ADDRESS,...,74 STREET,4013050133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# CELL 1: Memahami struktur dataset

print("Ukuran dataset:", df_nyc.shape)

print("\nDaftar kolom:")
print(df_nyc.columns.tolist())

print("\nInformasi dataset:")
df_nyc.info()

# Periksa kolom penting untuk rekomendasi
candidate_columns = [
    "unique_key",
    "created_date",
    "complaint_type",
    "descriptor",
    "descriptor_2",
    "agency_name",
    "resolution_description",
    "status",
    "borough"
]

available_columns = [
    col for col in candidate_columns
    if col in df_nyc.columns
]

display(df_nyc[available_columns].head(10))

Ukuran dataset: (1000, 46)

Daftar kolom:
['unique_key', 'created_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'incident_zip', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'facility_type', 'status', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'council_district', 'police_precinct', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'latitude', 'longitude', 'location', ':@computed_region_f5dn_yrer', ':@computed_region_yeji_bk3q', ':@computed_region_sbqj_enih', ':@computed_region_92fq_4b7q', 'location_type', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'landmark', 'bbl', 'vehicle_type', 'descriptor_2', 'taxi_pick_up_location', 'closed_date', 'bridge_highway_name', 'bridge_highway_direction', 'bridge_highway_segment', 'road_ramp']

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000

,unique_key,created_date,complaint_type,descriptor,descriptor_2,agency_name,resolution_description,status,borough
0,70709892,2026-10-11T02:42:35.000,Street Condition,Pothole,NaN,Department of Transportation,The Department of Transportation referred this...,Open,BROOKLYN
1,70720084,2026-10-11T01:51:04.000,Noise - Residential,Loud Music/Party,NaN,New York City Police Department,NaN,In Progress,Unspecified
2,70719843,2026-10-11T01:50:47.000,Noise - Residential,Loud Music/Party,NaN,New York City Police Department,NaN,In Progress,MANHATTAN
3,70719844,2026-10-11T01:50:36.000,Noise - Commercial,Loud Talking,NaN,New York City Police Department,NaN,In Progress,MANHATTAN
4,70720001,2026-10-11T01:50:36.000,Noise - Commercial,Loud Music/Party,NaN,New York City Police Department,NaN,In Progress,QUEENS
5,70720160,2026-10-11T01:50:35.000,Noise - Park,Loud Music/Party,NaN,New York City Police Department,NaN,In Progress,MANHATTAN
6,70720282,2026-10-11T01:50:00.000,Noise - Residential,Loud Music/Party,NaN,New York City Police Department,NaN,In Progress,QUEENS
7,70719466,2026-10-11T01:49:51.000,Illegal Parking,Blocked Hydrant,NaN,New York City Police Department,NaN,In Progress,BROOKLYN
8,70717329,2026-10-11T01:49:44.000,Noise - Residential,Banging/Pounding,NaN,New York City Police Department,NaN,In Progress,QUEENS
9,70714523,2026-10-11T01:49:42.000,Noise - Residential,Loud Music/Party,NaN,New York City Police Department,NaN,In Progress,MANHATTAN


### Data Preparation


In [11]:
# CELL 2: Data cleaning

import pandas as pd
import numpy as np

df = df_nyc.copy()

# Kolom teks yang tersedia
text_columns = [
    "complaint_type",
    "descriptor",
    "descriptor_2"
]

for col in text_columns:
    if col not in df.columns:
        df[col] = ""

    df[col] = df[col].fillna("").astype(str)

# Gabungkan informasi masalah
df["problem_text"] = (
    df["complaint_type"] + ". " +
    df["descriptor"] + ". " +
    df["descriptor_2"]
).str.strip()

# Hapus data tanpa informasi masalah
df = df[
    df["problem_text"].str.replace(".", "", regex=False).str.strip().ne("")
].copy()

df = df.reset_index(drop=True)

print("Jumlah laporan setelah cleaning:", len(df))
display(df[["problem_text"]].head(10))

Jumlah laporan setelah cleaning: 1000


,problem_text
0,Street Condition. Pothole.
1,Noise - Residential. Loud Music/Party.
2,Noise - Residential. Loud Music/Party.
3,Noise - Commercial. Loud Talking.
4,Noise - Commercial. Loud Music/Party.
5,Noise - Park. Loud Music/Party.
6,Noise - Residential. Loud Music/Party.
7,Illegal Parking. Blocked Hydrant.
8,Noise - Residential. Banging/Pounding.
9,Noise - Residential. Loud Music/Party.


## Buat Katalog Kegiatan

In [12]:
# CELL 3: Knowledge base kegiatan

activities = pd.DataFrame([
    {
        "activity": "Pemeriksaan dan perbaikan jalan",
        "problem": "Pothole, jalan rusak, permukaan jalan berlubang",
        "steps": [
            "Verifikasi lokasi dan kondisi jalan",
            "Dokumentasikan tingkat kerusakan",
            "Laporkan kepada instansi berwenang",
            "Lakukan perbaikan sesuai hasil pemeriksaan",
            "Evaluasi kondisi jalan setelah perbaikan"
        ],
        "responsible": "Instansi pemeliharaan jalan terkait",
        "indicator": "Kerusakan telah diperiksa dan ditindaklanjuti"
    },
    {
        "activity": "Investigasi gangguan kebisingan",
        "problem": "Noise, loud music, loud talking, party",
        "steps": [
            "Verifikasi waktu dan lokasi kejadian",
            "Catat jenis serta durasi gangguan",
            "Identifikasi instansi yang berwenang",
            "Lakukan tindak lanjut sesuai aturan setempat",
            "Pantau apakah gangguan berulang"
        ],
        "responsible": "Instansi pengelola pengaduan terkait",
        "indicator": "Laporan telah diverifikasi dan mendapat tindak lanjut"
    },
    {
        "activity": "Pemeriksaan kebersihan lingkungan",
        "problem": "Garbage, litter, trash, waste",
        "steps": [
            "Verifikasi lokasi penumpukan sampah",
            "Identifikasi sumber dan jenis sampah",
            "Koordinasikan penanganan dengan petugas terkait",
            "Lakukan pembersihan sesuai kewenangan",
            "Pantau kebersihan setelah penanganan"
        ],
        "responsible": "Instansi kebersihan atau pengelola lokasi",
        "indicator": "Sampah ditangani dan kondisi lokasi dievaluasi"
    },
    {
        "activity": "Pemeriksaan gangguan hama",
        "problem": "Rodent, rat, mouse, pest",
        "steps": [
            "Verifikasi lokasi dan indikasi gangguan hama",
            "Identifikasi faktor lingkungan yang mendukung",
            "Koordinasikan pemeriksaan oleh petugas kompeten",
            "Terapkan penanganan sesuai pedoman keselamatan",
            "Evaluasi apakah gangguan masih terjadi"
        ],
        "responsible": "Instansi kesehatan lingkungan terkait",
        "indicator": "Pemeriksaan dan tindak lanjut telah dievaluasi"
    }
])

display(activities[["activity", "problem", "responsible"]])

,activity,problem,responsible
0,Pemeriksaan dan perbaikan jalan,"Pothole, jalan rusak, permukaan jalan berlubang",Instansi pemeliharaan jalan terkait
1,Investigasi gangguan kebisingan,"Noise, loud music, loud talking, party",Instansi pengelola pengaduan terkait
2,Pemeriksaan kebersihan lingkungan,"Garbage, litter, trash, waste",Instansi kebersihan atau pengelola lokasi
3,Pemeriksaan gangguan hama,"Rodent, rat, mouse, pest",Instansi kesehatan lingkungan terkait


## Bangun Recommendation System dengan pretrained model

In [13]:
# CELL 4: Semantic recommendation engine

!pip -q install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Pretrained model
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Buat deskripsi lengkap setiap kegiatan
activities["search_text"] = (
    activities["activity"] + ". " +
    activities["problem"]
)

# Ubah katalog menjadi embeddings
activity_embeddings = model.encode(
    activities["search_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Recommendation engine siap!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Recommendation engine siap!


Kita buat fungsi rekomendasinya

In [14]:
# CELL 5: Fungsi rekomendasi

def recommend_activities(report_text, top_k=3):
    """
    Merekomendasikan kegiatan berdasarkan kemiripan
    semantik dengan laporan warga.
    """

    report_embedding = model.encode(
        [report_text],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        report_embedding,
        activity_embeddings
    )[0]

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in ranked_indices:
        item = activities.iloc[idx]

        results.append({
            "Kegiatan": item["activity"],
            "Skor kemiripan": round(float(scores[idx]), 3),
            "Langkah tindak lanjut": item["steps"],
            "Pihak terkait": item["responsible"],
            "Indikator hasil": item["indicator"]
        })

    return results

## Uji dengan laporan baru

In [19]:
# CELL 6: Simulasi laporan warga

report = """
Terdapat lubang besar di jalan yang membahayakan pengendara sepeda motor dan pengguna kendaraan lainnya. Jalan tersebut perlu diperiksa dan diperbaiki untuk mencegah kecelakaan.
"""

results = recommend_activities(report, top_k=2)

for i, result in enumerate(results, start=1):
    print(f"\nREKOMENDASI {i}")
    print("Kegiatan:", result["Kegiatan"])
    print("Skor kemiripan:", result["Skor kemiripan"])
    print("Pihak terkait:", result["Pihak terkait"])

    print("\nLangkah tindak lanjut:")
    for step_no, step in enumerate(
        result["Langkah tindak lanjut"], start=1
    ):
        print(f"{step_no}. {step}")

    print("Indikator:", result["Indikator hasil"])


REKOMENDASI 1
Kegiatan: Pemeriksaan dan perbaikan jalan
Skor kemiripan: 0.487
Pihak terkait: Instansi pemeliharaan jalan terkait

Langkah tindak lanjut:
1. Verifikasi lokasi dan kondisi jalan
2. Dokumentasikan tingkat kerusakan
3. Laporkan kepada instansi berwenang
4. Lakukan perbaikan sesuai hasil pemeriksaan
5. Evaluasi kondisi jalan setelah perbaikan
Indikator: Kerusakan telah diperiksa dan ditindaklanjuti

REKOMENDASI 2
Kegiatan: Pemeriksaan kebersihan lingkungan
Skor kemiripan: 0.424
Pihak terkait: Instansi kebersihan atau pengelola lokasi

Langkah tindak lanjut:
1. Verifikasi lokasi penumpukan sampah
2. Identifikasi sumber dan jenis sampah
3. Koordinasikan penanganan dengan petugas terkait
4. Lakukan pembersihan sesuai kewenangan
5. Pantau kebersihan setelah penanganan
Indikator: Sampah ditangani dan kondisi lokasi dievaluasi


## Hubungkan dengan laporan 311 NYC

In [17]:
# CELL 7: Pilih laporan asli dari dataset

display(
    df[
        [
            col for col in [
                "unique_key",
                "complaint_type",
                "descriptor",
                "problem_text"
            ]
            if col in df.columns
        ]
    ].head(10)
)

,unique_key,complaint_type,descriptor,problem_text
0,70709892,Street Condition,Pothole,Street Condition. Pothole.
1,70720084,Noise - Residential,Loud Music/Party,Noise - Residential. Loud Music/Party.
2,70719843,Noise - Residential,Loud Music/Party,Noise - Residential. Loud Music/Party.
3,70719844,Noise - Commercial,Loud Talking,Noise - Commercial. Loud Talking.
4,70720001,Noise - Commercial,Loud Music/Party,Noise - Commercial. Loud Music/Party.
5,70720160,Noise - Park,Loud Music/Party,Noise - Park. Loud Music/Party.
6,70720282,Noise - Residential,Loud Music/Party,Noise - Residential. Loud Music/Party.
7,70719466,Illegal Parking,Blocked Hydrant,Illegal Parking. Blocked Hydrant.
8,70717329,Noise - Residential,Banging/Pounding,Noise - Residential. Banging/Pounding.
9,70714523,Noise - Residential,Loud Music/Party,Noise - Residential. Loud Music/Party.


Lalu ambil 1 laporan

In [18]:
# Ambil satu laporan dari dataset
sample_report = df.loc[0, "problem_text"]

print("LAPORAN ASLI:")
print(sample_report)

print("\nREKOMENDASI AI:")

for result in recommend_activities(sample_report, top_k=3):
    print(
        f"- {result['Kegiatan']} "
        f"(similarity: {result['Skor kemiripan']})"
    )

LAPORAN ASLI:
Street Condition. Pothole.

REKOMENDASI AI:
- Pemeriksaan dan perbaikan jalan (similarity: 0.497)
- Pemeriksaan kebersihan lingkungan (similarity: 0.169)
- Investigasi gangguan kebisingan (similarity: 0.153)


### Apa yang sebaiknya kita kembangkan berikutnya?
Tahap 1: Prototype di Colab
Data cleaning, eksplorasi kategori, pretrained embeddings, dan rekomendasi kegiatan.


Tahap 2: Rekomendasi berdasarkan histori
Gunakan laporan serupa dan resolution_description jika tersedia untuk melihat tindakan yang pernah dilakukan. Tindakan historis tidak otomatis berarti tindakan itu efektif.


Tahap 3: Aplikasi interaktif
Buat UI untuk memasukkan laporan, menampilkan tiga rekomendasi, melihat langkah-langkah, dan memberi umpan balik.